# PathSnap — Model 1: Logistic Regression

Candidate classifier: each row is one (GPS observation, candidate road arc) pair with 24 features; the label is 1 if the arc is the observation's reference arc.

**What this notebook does**
1. Loads the features and labels produced by `Feature Extraction/extract_features.py`.
2. Runs 5-fold **GroupKFold by trip** on the train partition (76 usable trips).
3. Fits the final model on all usable train trips and evaluates on the 10 **student_test** trips.
4. Reports the metrics required by the Models and Training Guidelines, plus training and inference time.
5. Saves the model, the experiment log and the student_test probabilities (for the HMM stage).

**Before running on Colab:** upload `pathsnap_features.zip` to `MyDrive/PathSnap/` in Google Drive.

In [ ]:
# ---- Paths: Colab (Drive zip) or local run --------------------------------------------
import os, sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP = Path("/content/drive/MyDrive/PathSnap/pathsnap_features.zip")
    DATA = Path("/content/pathsnap_features")
    OUT = Path("/content/drive/MyDrive/PathSnap/results/logistic_regression")
    if not (DATA / "data_features").exists():
        with zipfile.ZipFile(ZIP) as z:
            z.extractall(DATA)
else:
    DATA = Path(os.environ.get("PATHSNAP_DATA", "../.."))
    OUT = Path(os.environ.get("PATHSNAP_OUT", "results/logistic_regression"))

FEAT_DIR = DATA / "data_features"
FLAGS = DATA / "trip_flags.csv" if (DATA / "trip_flags.csv").exists() else DATA / "data_clean" / "trip_flags.csv"
OUT.mkdir(parents=True, exist_ok=True)
print("features:", FEAT_DIR, "| flags:", FLAGS, "| output:", OUT)

In [ ]:
import json, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, log_loss, ConfusionMatrixDisplay)

SEED = 42
np.random.seed(SEED)

FEATURES = json.loads((FEAT_DIR / "feature_params.json").read_text())["FEATURES"]

CONFIG = {
    "model": "LogisticRegression",
    "family": "Linear and probabilistic",
    "scaler": "StandardScaler (fit on training folds only)",
    "C": 1.0,
    "penalty": "l2 (sklearn default; not passed, the argument is deprecated from 1.8)",
    "solver": "lbfgs",
    "max_iter": 2000,
    "class_weight": None,   # None keeps probabilities calibrated for the HMM; "balanced" is an option to compare
    "cv": "GroupKFold(n_splits=5) on trip_id",
    "seed": SEED,
    "n_features": len(FEATURES),
}

def make_model(cfg=CONFIG):
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=cfg["C"], solver=cfg["solver"],
                           max_iter=cfg["max_iter"], class_weight=cfg["class_weight"],
                           random_state=cfg["seed"]),
    )

print(f"sklearn {sklearn.__version__} | {len(FEATURES)} features")

## 1. Load data

* **Train:** only trips with `use_for_tuning = True` (trips 02, 33, 43, 54 have partial reference routes and are excluded).
* Rows with `label = -1` (outside the route's label span, or alignment failures) are dropped.
* Per-observation reference info (`*_points.csv.gz`) is loaded separately: it is needed to count observations whose reference arc has **no candidate row at all**, so top-1 accuracy is not inflated.

In [ ]:
flags = pd.read_csv(FLAGS, dtype={"trip_id": str})

def trips_of(part):
    t = flags[flags.partition == part]
    if part == "train":
        t = t[t.use_for_tuning.astype(bool)]
    return sorted(t.trip_id)

def load_partition(part):
    rows, pts = [], []
    for tid in trips_of(part):
        f = pd.read_csv(FEAT_DIR / part / f"{tid}.csv.gz", dtype={"trip_id": str})
        y = pd.read_csv(FEAT_DIR / "labels" / part / f"{tid}.csv.gz")
        assert len(f) == len(y) and (f.idx.values == y.idx.values).all() and (f.arc_id.values == y.arc_id.values).all()
        f["label"] = y.label.values
        rows.append(f[f.label >= 0])
        p = pd.read_csv(FEAT_DIR / "labels" / part / f"{tid}_points.csv.gz")
        p.insert(0, "trip_id", tid)
        pts.append(p[p.labelled])
    return pd.concat(rows, ignore_index=True), pd.concat(pts, ignore_index=True)

t0 = time.time()
train, train_pts = load_partition("train")
test, test_pts = load_partition("student_test")
print(f"loaded in {time.time() - t0:.1f}s")
for name, d, p in [("train", train, train_pts), ("student_test", test, test_pts)]:
    print(f"{name:13} trips {d.trip_id.nunique():3} | rows {len(d):>9,} | positive rate {d.label.mean():.3f} "
          f"| labelled observations {len(p):>7,} | candidate miss rate {1 - p.ref_in_candidates.mean():.4%}")

## 2. Metrics

**Row level** (threshold 0.5): accuracy, precision, recall, F1, confusion matrix, log loss. Only ~14% of rows are positive, so row accuracy is misleading on its own.

**Observation level** (the primary metric):
* **Top-1 accuracy** — for each labelled GPS observation, take the candidate with the highest score; correct if it is the reference arc. Observations with no correct candidate count as **wrong**.
* **Candidate miss rate** — share of labelled observations whose reference arc is not in the candidate set (an upper bound on achievable top-1). It does not depend on the model.

The nearest-arc rule (`dist_rank == 0`) is reported alongside as a non-ML reference.

In [ ]:
def row_metrics(y, p, thr=0.5):
    yhat = (p >= thr).astype(int)
    return {
        "accuracy": accuracy_score(y, yhat),
        "precision": precision_score(y, yhat, zero_division=0),
        "recall": recall_score(y, yhat, zero_division=0),
        "f1": f1_score(y, yhat, zero_division=0),
        "log_loss": log_loss(y, np.clip(p, 1e-7, 1 - 1e-7), labels=[0, 1]),
    }

def top1(df, score, pts):
    # top-1 over ALL labelled observations of the trips in df (no candidate = wrong)
    d = df[["trip_id", "idx", "label"]].assign(s=score)
    best = d.loc[d.groupby(["trip_id", "idx"]).s.idxmax()]
    hits = int(best.label.sum())
    n_obs = len(pts[pts.trip_id.isin(df.trip_id.unique())])
    n_reachable = int(pts[pts.trip_id.isin(df.trip_id.unique())].ref_in_candidates.sum())
    return {"top1": hits / n_obs, "top1_given_candidate": hits / max(n_reachable, 1), "n_obs": n_obs}

def evaluate(df, p, pts):
    m = row_metrics(df.label.values, p)
    m.update(top1(df, p, pts))
    nearest = top1(df, -df.dist_rank.values.astype(float), pts)
    m["top1_nearest_arc"] = nearest["top1"]
    return m

## 3. Cross-validation — GroupKFold by trip (train only)

All rows of a trip stay in the same fold: rows from one trip are strongly correlated, so a random row split would overstate accuracy. The scaler is fitted inside each fold on the training trips only.

In [ ]:
gkf = GroupKFold(n_splits=5)
cv_rows = []
for fold, (tr, va) in enumerate(gkf.split(train, train.label, groups=train.trip_id)):
    a, b = train.iloc[tr], train.iloc[va]
    model = make_model()
    t0 = time.time(); model.fit(a[FEATURES], a.label); fit_s = time.time() - t0
    t0 = time.time(); p = model.predict_proba(b[FEATURES])[:, 1]; pred_s = time.time() - t0
    m = evaluate(b, p, train_pts)
    m.update({"fold": fold, "train_trips": a.trip_id.nunique(), "val_trips": b.trip_id.nunique(),
              "fit_s": fit_s, "predict_s": pred_s, "predict_ms_per_1k_rows": 1000 * pred_s / len(b) * 1000})
    cv_rows.append(m)
    print(f"fold {fold}: top-1 {m['top1']:.4f} (nearest-arc {m['top1_nearest_arc']:.4f}) "
          f"| F1 {m['f1']:.4f} | log loss {m['log_loss']:.4f} | fit {fit_s:.1f}s")

cv = pd.DataFrame(cv_rows)
cols = ["top1", "top1_given_candidate", "top1_nearest_arc", "accuracy", "precision", "recall", "f1", "log_loss", "fit_s", "predict_s"]
cv_summary = cv[cols].agg(["mean", "std"]).T
cv_summary

## 4. Final model — fit on all usable train trips, evaluate on student_test

Student_test is used only to measure this finished model (development-stage comparison), never to choose hyperparameters.

In [ ]:
final = make_model()
t0 = time.time(); final.fit(train[FEATURES], train.label); final_fit_s = time.time() - t0
t0 = time.time(); p_test = final.predict_proba(test[FEATURES])[:, 1]; final_pred_s = time.time() - t0

test_metrics = evaluate(test, p_test, test_pts)
test_metrics.update({"fit_s": final_fit_s, "predict_s": final_pred_s, "rows": len(test)})
pd.Series(test_metrics).round(4)

In [ ]:
# Per-trip top-1 on student_test
per_trip = []
for tid, g in test.assign(p=p_test).groupby("trip_id"):
    per_trip.append({"trip_id": tid, **top1(g, g.p.values, test_pts),
                     "top1_nearest_arc": top1(g, -g.dist_rank.values.astype(float), test_pts)["top1"]})
per_trip = pd.DataFrame(per_trip).sort_values("top1")
per_trip.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(test.label, (p_test >= 0.5).astype(int))
ConfusionMatrixDisplay(cm, display_labels=["not ref (0)", "ref arc (1)"]).plot(ax=axes[0], colorbar=False, values_format=",")
axes[0].set_title("Student_test confusion matrix (rows, threshold 0.5)")

coef = pd.Series(final[-1].coef_[0], index=FEATURES).sort_values()
axes[1].barh(coef.index, coef.values, color=np.where(coef.values >= 0, "#2a78d6", "#eb6834"))
axes[1].axvline(0, color="#9c9b96", lw=1)
axes[1].set_title("Standardised coefficients (positive = more likely the reference arc)")
axes[1].tick_params(axis="y", labelsize=8)
fig.tight_layout()
fig.savefig(OUT / "logreg_confusion_and_coefficients.png", dpi=150)
plt.show()

## 5. Save model, experiment log and predictions

* `logreg_model.joblib` — the fitted pipeline (scaler + model).
* `experiment_log.json` — configuration, seed, versions, per-fold CV results, student_test results and timings.
* `student_test_predictions.csv.gz` — per-candidate probabilities, to be used as emission scores in the HMM/Viterbi stage.

In [ ]:
joblib.dump(final, OUT / "logreg_model.joblib")

log = {
    "config": CONFIG,
    "features": FEATURES,
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__,
                 "numpy": np.__version__, "pandas": pd.__version__},
    "data": {"train_trips": trips_of("train"), "student_test_trips": trips_of("student_test"),
             "train_rows": len(train), "student_test_rows": len(test)},
    "cv_folds": cv.to_dict(orient="records"),
    "cv_mean": cv[cols].mean().to_dict(),
    "cv_std": cv[cols].std().to_dict(),
    "student_test": test_metrics,
    "student_test_per_trip": per_trip.to_dict(orient="records"),
}
(OUT / "experiment_log.json").write_text(json.dumps(log, indent=2, default=float))

test[["trip_id", "idx", "arc_id", "label"]].assign(p_logreg=p_test).to_csv(
    OUT / "student_test_predictions.csv.gz", index=False, float_format="%.6f")
print("saved to", OUT)